# Модель

In [1]:
!pip install catboost

In [2]:
import pandas as pd
import numpy as np
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Загрузим данные
df = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\ML\train.csv')
df['snapshot_date'] = pd.to_datetime(df['snapshot_date'])

In [28]:
df.head()

,customer_id,snapshot_date,dpd_bucket,outstanding_amount,income,region,product_type,contacts_last_7d,rpc_last_30d,promises_last_30d,payments_amount_last_30d,target_paid_30d
0,100056,2025-10-01,1-30,10987.42,NaN,Siberia,micro_loan,3,0,1,0.00,0
1,100110,2025-10-01,61-90,30961.88,75940.84,Ural,installment,3,1,0,0.00,0
2,100111,2025-10-01,31-60,29000.38,82695.80,Center,installment,3,2,1,4746.34,1
3,100130,2025-10-01,1-30,21187.05,107453.10,Center,micro_loan,1,1,1,4280.34,1
4,100285,2025-10-01,1-30,33901.72,58239.00,South,micro_loan,1,0,0,2275.41,0


In [3]:
# Разобьем данные на тренировочные и валидационные
train_mask = (df['snapshot_date'] >= '2025-10-01') & (df['snapshot_date'] <= '2025-12-31')
val_mask = (df['snapshot_date'] >= '2026-01-01') & (df['snapshot_date'] <= '2026-01-31')

df_train = df.loc[train_mask].copy()
df_val = df.loc[val_mask].copy()

target_col = 'target_paid_30d'
drop_cols = ['customer_id', 'snapshot_date', target_col]

X_train = df_train.drop(columns=drop_cols)
y_train = df_train[target_col]

X_val = df_val.drop(columns=drop_cols)
y_val = df_val[target_col]

In [4]:
# Подготовим и обучим модель CatBoost
cat_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

model_cb = CatBoostClassifier(
    iterations=300,
    depth=4,
    l2_leaf_reg=1,
    learning_rate=0.05,
    verbose=False,
    random_state=42
)

model_cb.fit(
    X_train,
    y_train,
    cat_features=cat_features,
    eval_set=(X_val, y_val),
    use_best_model=True,
    early_stopping_rounds=200
)

val_pred_cb = model_cb.predict_proba(X_val)[:, 1]

In [5]:
# Подготовим и обучим логистическую регрессию
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
numeric_features = X_train.select_dtypes(include=[np.number, 'bool']).columns.tolist()

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model_lr = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])

model_lr.fit(X_train, y_train)
val_pred_lr = model_lr.predict_proba(X_val)[:, 1]

In [6]:
# Вычислим необходимые метрики
def calc_metrics(y_true, y_pred, df_part):
    roc = roc_auc_score(y_true, y_pred)
    pr = average_precision_score(y_true, y_pred)

    df_tmp = df_part.copy()
    df_tmp['pred'] = y_pred

    top_k = int(len(df_tmp) * 0.30)
    top = df_tmp.sort_values('pred', ascending=False).head(top_k)

    precision_top30 = top[target_col].mean()

    return roc, pr, precision_top30

roc_cb, pr_cb, p30_cb = calc_metrics(y_val, val_pred_cb, df_val)
roc_lr, pr_lr, p30_lr = calc_metrics(y_val, val_pred_lr, df_val)

In [7]:
# Посчитаем изменение общих метрик
delta_df = pd.DataFrame({
    'metric': ['roc_auc', 'pr_auc', 'precision_top30'],
    'logreg': [roc_lr, pr_lr, p30_lr],
    'catboost': [roc_cb, pr_cb, p30_cb]
})

delta_df['delta'] = delta_df['catboost'] - delta_df['logreg']
delta_df['trend'] = np.where(delta_df['delta'] > 0, 'выросла', np.where(delta_df['delta'] < 0, 'упала', 'без изменений'))

print('\nИзменение метрик: CatBoost vs LogReg')
print(delta_df.round(4))


Изменение метрик: CatBoost vs LogReg
            metric  logreg  catboost   delta    trend
0          roc_auc  0.7894    0.7886 -0.0008    упала
1           pr_auc  0.7191    0.7155 -0.0036    упала
2  precision_top30  0.7031    0.7088  0.0057  выросла


In [8]:
# Определим топ-10 важных признаков
feature_importance = pd.DataFrame({'feature': X_train.columns, 'importance': model_cb.get_feature_importance()}).sort_values('importance', ascending=False)

print('\nToп-10 наиболее важных признаков для CatBoost')
print(feature_importance.head(10).round(4))


Toп-10 наиболее важных признаков для CatBoost
                    feature  importance
0                dpd_bucket     20.2631
7         promises_last_30d     15.8341
1        outstanding_amount     15.7017
6              rpc_last_30d     15.6857
5          contacts_last_7d      8.5109
4              product_type      7.1608
2                    income      6.6294
8  payments_amount_last_30d      5.7038
3                    region      4.5105


In [9]:
# Посчитаем метрики по сегментам
def segment_metrics(df, pred_lr, pred_cb, segment_col):
    df_tmp = df.copy()
    df_tmp['pred_lr'] = pred_lr
    df_tmp['pred_cb'] = pred_cb

    result = []

    for val in df_tmp[segment_col].unique():
        part = df_tmp[df_tmp[segment_col] == val]

        if len(part) < 50:
            continue

        def calc(pred):
            roc = roc_auc_score(part[target_col], pred)
            pr = average_precision_score(part[target_col], pred)

            top_k = int(len(part) * 0.30)
            top = part.assign(p=pred).sort_values('p', ascending=False).head(top_k)
            p30 = top[target_col].mean()

            return roc, pr, p30

        roc_lr_s, pr_lr_s, p30_lr_s = calc(part['pred_lr'])
        roc_cb_s, pr_cb_s, p30_cb_s = calc(part['pred_cb'])

        result.append({segment_col: val, 'size': len(part), 'roc_lr': roc_lr_s, 'roc_cb': roc_cb_s, 'roc_delta': roc_cb_s - roc_lr_s, 
                       'pr_lr': pr_lr_s, 'pr_cb': pr_cb_s, 'pr_delta': pr_cb_s - pr_lr_s, 'p30_lr': p30_lr_s, 'p30_cb': p30_cb_s, 
                       'p30_delta': p30_cb_s - p30_lr_s})

    return pd.DataFrame(result)

dpd_metrics = segment_metrics(df_val, val_pred_lr, val_pred_cb, 'dpd_bucket')
region_metrics = segment_metrics(df_val, val_pred_lr, val_pred_cb, 'region')

In [10]:
# Посчтиаем отдельно метрики по `dpd_bucket`
dpd_roc = dpd_metrics[['dpd_bucket', 'size', 'roc_lr', 'roc_cb', 'roc_delta']].copy()
dpd_roc['trend'] = np.where(dpd_roc['roc_delta'] > 0, 'выросла', np.where(dpd_roc['roc_delta'] < 0, 'упала', 'без изменений'))

dpd_pr = dpd_metrics[['dpd_bucket', 'size', 'pr_lr', 'pr_cb', 'pr_delta']].copy()
dpd_pr['trend'] = np.where(dpd_pr['pr_delta'] > 0, 'выросла', np.where(dpd_pr['pr_delta'] < 0, 'упала', 'без изменений'))

dpd_p30 = dpd_metrics[['dpd_bucket', 'size', 'p30_lr', 'p30_cb', 'p30_delta']].copy()
dpd_p30['trend'] = np.where(dpd_p30['p30_delta'] > 0, 'выросла', np.where(dpd_p30['p30_delta'] < 0, 'упала', 'без изменений'))

print('\ndpd_bucket: ROC-AUC')
print(dpd_roc.round(4))

print('\ndpd_bucket: PR-AUC')
print(dpd_pr.round(4))

print('\ndpd_bucket: precision@top30%')
print(dpd_p30.round(4))


dpd_bucket: ROC-AUC
  dpd_bucket  size  roc_lr  roc_cb  roc_delta    trend
0        90+   205  0.7859  0.7651    -0.0208    упала
1      61-90   372  0.7935  0.7955     0.0020  выросла
2      31-60   473  0.7726  0.7711    -0.0015    упала
3       1-30   691  0.7613  0.7634     0.0021  выросла

dpd_bucket: PR-AUC
  dpd_bucket  size   pr_lr   pr_cb  pr_delta    trend
0        90+   205  0.5536  0.4870   -0.0666    упала
1      61-90   372  0.6560  0.6624    0.0064  выросла
2      31-60   473  0.6713  0.6725    0.0012  выросла
3       1-30   691  0.7795  0.7756   -0.0039    упала

dpd_bucket: precision@top30%
  dpd_bucket  size  p30_lr  p30_cb  p30_delta          trend
0        90+   205  0.4918  0.4590    -0.0328          упала
1      61-90   372  0.6306  0.6126    -0.0180          упала
2      31-60   473  0.6809  0.6667    -0.0142          упала
3       1-30   691  0.7874  0.7874     0.0000  без изменений


In [11]:
# Посчитаем отдельно метрики по `region`
region_roc = region_metrics[['region', 'size', 'roc_lr', 'roc_cb', 'roc_delta']].copy()
region_roc['trend'] = np.where(region_roc['roc_delta'] > 0, 'выросла', np.where(region_roc['roc_delta'] < 0, 'упала', 'без изменений'))

region_pr = region_metrics[['region', 'size', 'pr_lr', 'pr_cb', 'pr_delta']].copy()
region_pr['trend'] = np.where(region_pr['pr_delta'] > 0, 'выросла', np.where(region_pr['pr_delta'] < 0, 'упала', 'без изменений'))

region_p30 = region_metrics[['region', 'size', 'p30_lr', 'p30_cb', 'p30_delta']].copy()
region_p30['trend'] = np.where(region_p30['p30_delta'] > 0, 'выросла', np.where(region_p30['p30_delta'] < 0, 'упала', 'без изменений'))

print('\nregion: ROC-AUC')
print(region_roc.round(4))

print('\nregion: PR-AUC')
print(region_pr.round(4))

print('\nregion: precision@top30%')
print(region_p30.round(4))


region: ROC-AUC
    region  size  roc_lr  roc_cb  roc_delta    trend
0   Center   347  0.7597  0.7619     0.0022  выросла
1   Moscow   415  0.7648  0.7679     0.0032  выросла
2      SPB   257  0.7927  0.7885    -0.0042    упала
3     Ural   229  0.8420  0.8331    -0.0089    упала
4    South   301  0.8216  0.8153    -0.0063    упала
5  Siberia   192  0.7877  0.7861    -0.0016    упала

region: PR-AUC
    region  size   pr_lr   pr_cb  pr_delta    trend
0   Center   347  0.7093  0.7050   -0.0043    упала
1   Moscow   415  0.6986  0.7007    0.0021  выросла
2      SPB   257  0.7255  0.7133   -0.0121    упала
3     Ural   229  0.7927  0.7856   -0.0071    упала
4    South   301  0.7418  0.7328   -0.0091    упала
5  Siberia   192  0.7278  0.7239   -0.0039    упала

region: precision@top30%
    region  size  p30_lr  p30_cb  p30_delta          trend
0   Center   347  0.7019  0.6731    -0.0288          упала
1   Moscow   415  0.7177  0.7177     0.0000  без изменений
2      SPB   257  0.6883  0.6

# Вывод

Модель CatBoost не показала улучшения по сравнению с baseline на Logistic Regression: ROC-AUC и PR-AUC немного снизились, однако ключевая бизнес-метрика precision@top30% выросла, что говорит о лучшем ранжировании клиентов в топ-сегменте. 

При этом на уровне сегментов наблюдается деградация по dpd_bucket — особенно в группе 90+, где ухудшились все метрики, включая precision@top30%. 

По регионам результаты неоднородны: есть улучшения в отдельных сегментах (например, Ural и South по precision), но в большинстве регионов качество снизилось. 

Таким образом, бустинг даёт локальные улучшения, но в целом уступает baseline по стабильности и требует дополнительного тюнинга или доработки признаков.